# 長期記憶の有無によるエージェントの比較検証

このJupyter Notebookでは、「長期記憶を持つエージェント」と「持たないエージェント」の挙動の違いを明確に比較します。

「セッションを再起動し、新しいエージェントのインスタンスを作成する」という操作をシミュレートすることで、「まるで1日後に話しかけた」ような状況を作り出し、記憶が永続化されているかを確認します。

## 共通セットアップ
まず、両方の実験で必要となるライブラリをインストールし、インポートします。

In [ ]:
!pip install -q boto3 strands bedrock-agentcore

In [ ]:
import logging
from datetime import datetime
from botocore.exceptions import ClientError

from strands import Agent
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("memory-comparison")

---

## パート1: 長期記憶を持つエージェント (AgentCore Memoryを使用)

このエージェントは、会話の履歴を永続的なストレージに保存するため、セッションを越えてユーザーの情報を記憶できます。

In [ ]:
# パート1用の設定
REGION = "us-west-2"  # ご自身のリージョンに合わせて変更してください
USER_ID = "user_long_term_memory_test"
MEMORY_NAME = "LongTermMemoryForAgent"

In [ ]:
# メモリクライアントとメモリリソースを作成
memory_client = MemoryClient(region_name=REGION)
strategies = [{
    StrategyType.USER_PREFERENCE.value: {
        "name": "UserPreferences", "description": "ユーザーの好みを記録", "namespaces": ["user/{actorId}/preferences"]
    }
}]

memory_id = None
try:
    memory = memory_client.create_memory_and_wait(name=MEMORY_NAME, strategies=strategies)
    memory_id = memory['id']
    logger.info(f"✅ メモリを作成しました: {memory_id}")
except ClientError as e:
    if "already exists" in str(e):
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(MEMORY_NAME)), None)
        logger.info(f"メモリは既に存在します。既存のIDを使用します: {memory_id}")
    else: raise e

In [ ]:
# 長期記憶用のフッククラス
class LongTermMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id, self.client, self.actor_id, self.session_id = memory_id, client, actor_id, session_id
        strategies = self.client.get_memory_strategies(self.memory_id)
        self.namespace = strategies[0]["namespaces"][0]
    def retrieve_context(self, event: MessageAddedEvent):
        if event.agent.messages[-1]["role"] == "user":
            user_query = event.agent.messages[-1]["content"][0]["text"]
            memories = self.client.retrieve_memories(memory_id=self.memory_id, namespace=self.namespace.format(actorId=self.actor_id), query=user_query)
            if memories:
                context = "\n".join([m.get('content', {}).get('text', '') for m in memories])
                event.agent.messages[-1]["content"][0]["text"] = f"以前の文脈:\n{context}\n\n現在の質問: {user_query}"
    def save_conversation(self, event: AfterInvocationEvent):
        if len(event.agent.messages) >= 2:
             user_msg, assistant_msg = event.agent.messages[-2:]
             self.client.create_event(memory_id=self.memory_id, actor_id=self.actor_id, session_id=self.session_id, messages=[(user_msg['content'][0]['text'], "USER"), (assistant_msg['content'][0]['text'], "ASSISTANT")])
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(MessageAddedEvent, self.retrieve_context)
        registry.add_callback(AfterInvocationEvent, self.save_conversation)

### 1-1. 最初の会話：好きな食べ物を教える

In [ ]:
# 最初のセッション
session_id_1 = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# エージェントを作成し、好きな食べ物を教える
agent_remembers_1 = Agent(
    hooks=[LongTermMemoryHooks(memory_id, memory_client, USER_ID, session_id_1)],
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    system_prompt="あなたは雑談ができるアシスタントです。"
)

response = agent_remembers_1("私の好きな食べ物はカレーだよ！覚えておいてね")

### 1-2. 日を改めて...：新しいセッションで質問する
**ここが重要です。** `Agent` を新しく作り直します。これは、PCを再起動したり、次の日に再度プログラムを実行したりするのと同じです。しかし、同じ `memory_id` に接続することで、過去の記憶を引き継ぎます。

In [ ]:
# 新しいセッションIDで、新しいエージェントのインスタンスを作成
session_id_2 = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}_new"

agent_remembers_2 = Agent(
    hooks=[LongTermMemoryHooks(memory_id, memory_client, USER_ID, session_id_2)],
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    system_prompt="あなたは雑談ができるアシスタントです。"
)

response = agent_remembers_2("こんにちは。ところで、私の好きな食べ物、覚えていますか？")

---

## パート2: 長期記憶を持たないエージェント (ステートレス)

このエージェントは外部の記憶システムに接続されていません。そのため、新しいインスタンスが作成されると、それ以前の会話はすべて忘れてしまいます。

### 2-1. 最初の会話：好きな食べ物を教える

In [ ]:
# 最初のエージェントインスタンスを作成し、好きな食べ物を教える
agent_forgets_1 = Agent(
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    system_prompt="あなたは雑談ができるアシスタントです。"
)

response = agent_forgets_1("私の好きな食べ物はハンバーグだよ！覚えておいてね")

### 2-2. 日を改めて...：新しいセッションで質問する
こちらでも同様に、新しい `Agent` を作り直します。

In [ ]:
# 新しいエージェントのインスタンスを作成
agent_forgets_2 = Agent(
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    system_prompt="あなたは雑談ができるアシスタントです。"
)

response = agent_forgets_2("こんにちは。ところで、私の好きな食べ物、覚えていますか？")